In [46]:
import pandas as pd
import numpy as np
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding, EarlyStoppingCallback
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch import nn
from transformers import logging as hf_logging
import json

hf_logging.set_verbosity_error()

np.random.seed(42)
torch.manual_seed(42)


In [47]:
# Load data
train_df = pd.read_csv('llm_annotated_train_set.csv')
test_df = pd.read_csv('talk_moves_validation_set.csv')

# Helper functions for data processing
def get_turn_num(row):
    if pd.notna(row['turn']):
        try:
            return int(float(row['turn']))
        except ValueError:
            return 'nan'
    return 'nan'

def parse_context_utterances(context_str):
    """Parse '(x) [S/T] text\n(x) [S/T] text' into list of individual utterance strings."""
    if pd.isna(context_str) or context_str.strip() == '':
        return []
    return [line.strip() for line in context_str.strip().split('\n') if line.strip()]

def create_windowed_input(row, tokenizer, window_size=7, utterance_max_tokens=9999):
    """
    Build input as a list of utterance strings, each truncated to utterance_max_tokens.
    Structure: [prev_1, ..., prev_n, [TARGET_START] target [TARGET_END], subseq_1, ..., subseq_n]
    These will be joined with </s></s> separators by the tokenizer.
    """
    turn_num = get_turn_num(row)

    prev_utterances  = parse_context_utterances(row['previous_context'])[-window_size:]
    subseq_utterances = parse_context_utterances(row['subsequent_context'])[:window_size]
    target = f"[TARGET_START] ({turn_num}) [S] {row['student_utterance']} [TARGET_END]"

    all_utterances = prev_utterances + [target] + subseq_utterances

    # Truncate each utterance individually to utterance_max_tokens
    truncated = []
    for utt in all_utterances:
        tokens = tokenizer.tokenize(utt)[:utterance_max_tokens]
        truncated.append(tokenizer.convert_tokens_to_string(tokens))

    return truncated

def make_dataset(df, tokenizer, window_size=7, utterance_max_tokens=9999):
    utterances = df.apply(
        lambda row: create_windowed_input(row, tokenizer, window_size, utterance_max_tokens),
        axis=1
    ).tolist()
    return Dataset.from_dict({
        'utterances': utterances,
        'label': df['label'].tolist()
    })


In [48]:
# Custom Trainer with class-weighted loss
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self.class_weights.to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'f1': f1_score(labels, predictions, zero_division=0)
    }


In [50]:
def run_model(label, window_size=7, utterance_max_tokens=9999):
    np.random.seed(42)
    torch.manual_seed(42)
    print(f"\n{'='*60}")
    print(f"Training model for: {label}")
    print(f"Architecture 2: windowed ({window_size} prev + target + {window_size} subseq), {utterance_max_tokens} tokens/utterance")
    print(f"{'='*60}")

    # Calculate class weights
    neg_count = (train_df[label] == 0).sum()
    pos_count = (train_df[label] == 1).sum()
    pos_weight = neg_count / pos_count if pos_count > 0 else 1.0
    print(f"Class weight for positive class: {pos_weight:.2f}")
    class_weights = torch.tensor([1.0, pos_weight], dtype=torch.float)

    print(f"Test set size: {len(test_df)}")
    print(f"Positive examples in test: {test_df[label].sum()}")
    print(f"Negative examples in test: {(test_df[label]==0).sum()}")
    print(f"Positive ratio in test: {test_df[label].mean():.3f}")

    print(f"\nTrain set size: {len(train_df)}")
    print(f"Positive examples in train: {train_df[label].sum()}")
    print(f"Positive ratio in train: {train_df[label].mean():.3f}")

    # Load model and tokenizer
    model_name = "roberta-base"
    tokenizer = RobertaTokenizer.from_pretrained(model_name)

    # Add special tokens for target utterance boundary
    special_tokens = {'additional_special_tokens': ['[TARGET_START]', '[TARGET_END]']}
    tokenizer.add_special_tokens(special_tokens)

    model = RobertaForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )
    model.resize_token_embeddings(len(tokenizer))

    # Prepare datasets
    train_data_label = train_df[['previous_context', 'student_utterance', 'turn', 'subsequent_context', label]].copy()
    train_data_label.columns = ['previous_context', 'student_utterance', 'turn', 'subsequent_context', 'label']

    test_data_label = test_df[['previous_context', 'student_utterance', 'turn', 'subsequent_context', label]].copy()
    test_data_label.columns = ['previous_context', 'student_utterance', 'turn', 'subsequent_context', 'label']

    train_dataset = make_dataset(train_data_label, tokenizer, window_size, utterance_max_tokens)
    test_dataset  = make_dataset(test_data_label,  tokenizer, window_size, utterance_max_tokens)


    all_utterances = []
    for utt_list in train_dataset['utterances']:
        all_utterances.extend(utt_list)

    lengths = pd.Series([len(tokenizer.tokenize(u)) for u in all_utterances])
    print(lengths.describe())
    print(f"95th percentile: {lengths.quantile(0.95):.0f} tokens")
    print(f"99th percentile: {lengths.quantile(0.99):.0f} tokens")


    # Check sequence lengths before tokenizing
    def get_full_length(utterances):
        input_ids = [tokenizer.bos_token_id]
        for i, utt in enumerate(utterances):
            input_ids += tokenizer.encode(utt, add_special_tokens=False)
            if i < len(utterances) - 1:
                input_ids += [tokenizer.eos_token_id, tokenizer.eos_token_id]
        input_ids += [tokenizer.eos_token_id]
        return len(input_ids)

    lengths = pd.Series([get_full_length(u) for u in train_dataset['utterances']])
    print(f"\nSequence length statistics:")
    print(lengths.describe())
    print(f"Truncated (>512): {(lengths > 512).sum()} / {len(lengths)} ({100*(lengths > 512).mean():.1f}%)")

    # Tokenize: each utterance gets its own </s></s> boundary
    def tokenize_windowed(examples):
        batch_input_ids = []
        batch_attention_masks = []

        for utterances in examples['utterances']:
            # Build: <s> utt1 </s></s> utt2 </s></s> ... uttN </s>
            input_ids = [tokenizer.bos_token_id]
            for i, utt in enumerate(utterances):
                input_ids += tokenizer.encode(utt, add_special_tokens=False)
                if i < len(utterances) - 1:
                    input_ids += [tokenizer.eos_token_id, tokenizer.eos_token_id]
            input_ids += [tokenizer.eos_token_id]

            # Truncate to 512
            input_ids = input_ids[:512]
            attention_mask = [1] * len(input_ids)

            batch_input_ids.append(input_ids)
            batch_attention_masks.append(attention_mask)

        return {
            'input_ids': batch_input_ids,
            'attention_mask': batch_attention_masks
        }

    train_dataset = train_dataset.map(tokenize_windowed, batched=True)
    test_dataset  = test_dataset.map(tokenize_windowed, batched=True)

    # Use DataCollatorWithPadding for dynamic padding per batch
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

    # Set up training arguments
    label_safe = label.replace(' ', '_')
    training_args = TrainingArguments(
        output_dir=f'./roberta_arch2_{label_safe}',
        num_train_epochs=5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        warmup_ratio=0.1,
        weight_decay=0.01,
        learning_rate=2e-5,
        logging_dir=f'./logs_arch2_{label_safe}',
        logging_steps=50,
        logging_strategy='epoch',
        report_to='none',
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        greater_is_better=True
    )

    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
        data_collator=data_collator
    )

    print(f"\nStarting training for {label}...")
    trainer.train()

    history = trainer.state.log_history
    history_df = pd.DataFrame(history)
    history_df.to_csv(f'training_history_arch2_{label_safe}.csv', index=False)

    print(f"\nEvaluating on test set...")
    results = trainer.evaluate(test_dataset)
    print(f"Test Results for {label}:")
    for key, value in results.items():
        print(f"  {key}: {value:.4f}")

    with open(f'results_arch2_{label_safe}.json', 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Metrics saved to results_arch2_{label_safe}.json")

    # Get and save predictions
    predictions = trainer.predict(test_dataset)
    pred_labels = np.argmax(predictions.predictions, axis=1)
    test_data_label['predicted_label'] = pred_labels
    test_data_label.to_csv(f'test_predictions_arch2_{label_safe}.csv', index=False)

    model.save_pretrained(f'./roberta_arch2_{label_safe}')
    tokenizer.save_pretrained(f'./roberta_arch2_{label_safe}')
    print(f"\nCompleted training for {label}.\n")


In [51]:
labels = ['Offering Math Help', 'Successful Uptake']

for label in labels:

    run_model(label)



Training model for: Offering Math Help
Architecture 2: windowed (7 prev + target + 7 subseq), 9999 tokens/utterance
Class weight for positive class: 2.58
Test set size: 180
Positive examples in test: 16
Negative examples in test: 164
Positive ratio in test: 0.089

Train set size: 2500
Positive examples in train: 698
Positive ratio in train: 0.279


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

count    37500.000000
mean        15.259467
std          8.156512
min          2.000000
25%         10.000000
50%         13.000000
75%         18.000000
max        223.000000
dtype: float64
95th percentile: 30 tokens
99th percentile: 45 tokens

Sequence length statistics:
count    2500.000000
mean      258.892000
std        46.649712
min       168.000000
25%       227.000000
50%       252.000000
75%       281.000000
max       550.000000
dtype: float64
Truncated (>512): 2 / 2500 (0.1%)


Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

...

{'loss': '0.3119', 'grad_norm': '111', 'learning_rate': '1.42e-08', 'epoch': '5'}
{'eval_loss': '0.9045', 'eval_accuracy': '0.6667', 'eval_precision': '0.4545', 'eval_recall': '0.8824', 'eval_f1': '0.6', 'eval_runtime': '3.309', 'eval_samples_per_second': '54.4', 'eval_steps_per_second': '6.951', 'epoch': '5'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '1174', 'train_samples_per_second': '10.64', 'train_steps_per_second': '1.333', 'train_loss': '0.5039', 'epoch': '5'}

Evaluating on test set...
{'eval_loss': '0.5494', 'eval_accuracy': '0.6889', 'eval_precision': '0.4742', 'eval_recall': '0.902', 'eval_f1': '0.6216', 'eval_runtime': '3.316', 'eval_samples_per_second': '54.28', 'eval_steps_per_second': '6.935', 'epoch': '5'}
Test Results for Successful Uptake:
  eval_loss: 0.5494
  eval_accuracy: 0.6889
  eval_precision: 0.4742
  eval_recall: 0.9020
  eval_f1: 0.6216
  eval_runtime: 3.3163
  eval_samples_per_second: 54.2770
  eval_steps_per_second: 6.9350
  epoch: 5.0000
Metrics saved to results_arch2_Successful_Uptake.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Completed training for Successful Uptake.

